In [44]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [45]:
pd.set_option('display.max_columns', None)

In [46]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

2025-03-02 11:42:45,078 - asyncio - ERROR - Task exception was never retrieved
future: <Task finished name='Task-3' coro=<process_medical_data() done, defined at /var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_82408/1646644219.py:352> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/Users/nam-yeong/miniforge3/envs/pymc_env/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_82408/1646644219.py", line 439, in <module>
    main()
  File "/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_82408/1646644219.py", line 412, in main
    processed_df = loop.run_until_complete(process_medical_data(df_sample, api_key))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nam-yeong/miniforge3/envs/pymc_env/lib/python3.12/site-packages/nest_asyncio.py", l

In [47]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']]

In [5]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI']]

In [57]:
df = df.sample(100)

In [59]:
# integrated_medical_classifier.py
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import anthropic
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
from prompts.medical_prompts import PresentIllnessPrompts
from tqdm.asyncio import tqdm as tqdm_asyncio

# 설정 클래스
class Config:
    API_KEY = api_key
    MODEL_NAME = "claude-3-sonnet-20240229"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 50
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 3
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"


# 로깅 설정 함수
def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)


# 프롬프트 클래스들 정의 - 생략...


#############################################
# 체크포인트 관리 클래스
#############################################

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def get_checkpoint_path(self, column: str) -> str:
        safe_column = column.replace("/", "_").replace("\\", "_")
        return os.path.join(self.checkpoint_dir, f"{safe_column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")

    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                return pd.read_parquet(path)
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None


#############################################
# 메디컬 텍스트 분류기 클래스
#############################################

class MedicalTextClassifier:
    """의학 텍스트 분류기 클래스"""
    
    def __init__(self, api_key: str, config=None):
        """초기화"""
        self.client = anthropic.Anthropic(api_key=api_key)
        self.config = config if config is not None else Config
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager(self.config.CHECKPOINT_DIR)
        
        # 분류기 메소드 맵핑
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            # 'PI': self._classify_present_illness,
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리"""
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                df = await self._process_column_with_checkpoint(df, column)
        return df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트를 사용한 컬럼 처리"""
        try:
            # 기존 체크포인트 확인
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                df.update(checkpoint_df)
                logger.info(f"Resumed from checkpoint for {column}")
                return df

            # 유효한 텍스트 처리
            mask = df[column].notna() & df[column].str.strip().astype(bool)
            if not mask.any():
                return df

            texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
            
            # 데이터 처리
            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            if results:
                result_df = pd.DataFrame(results).set_index('index')
                for col in result_df.columns:
                    new_col = f"{column}_{col}"
                    df[new_col] = result_df[col]

            self._cleanup_checkpoint(column)
            return df

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise

    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                  classifier, column: str) -> List[Dict]:
        """안전한 배치 처리"""
        results = []
        batch_size = self.config.BATCH_SIZE

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_indices = original_indices[i:i+batch_size]

            try:
                batch_results = await self._process_with_retry(
                    classifier, batch_texts, batch_indices
                )
                results.extend(batch_results)

                # 부분 결과 저장
                partial_df = pd.DataFrame(batch_results).set_index('index')
                self.checkpoint.save_checkpoint(partial_df, column)

            except Exception as e:
                logger.error(f"Batch {i//batch_size} failed: {str(e)}")
                continue

        return results

    @retry(stop=stop_after_attempt(3),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _process_with_retry(self, classifier, batch_texts: List[str],
                                batch_indices: List[int]) -> List[Dict]:
        """재시도 로직이 있는 처리"""
        async with self.semaphore:
            results = await classifier(batch_texts, self.semaphore)
            return [{"index": idx, **res} for idx, res in zip(batch_indices, results)]

    def _cleanup_checkpoint(self, column: str) -> None:
        """성공적인 처리 후 체크포인트 정리"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    @retry(stop=stop_after_attempt(3),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 호출 메소드"""
        try:
            async with semaphore:
                response = await asyncio.to_thread(
                    self.client.messages.create,
                    model=self.config.MODEL_NAME,
                    max_tokens=self.config.MAX_TOKENS,
                    temperature=self.config.TEMPERATURE,
                    system="JSON 형식으로 응답하세요.",
                    messages=[{"role": "user", "content": prompt}]
                )

                content = response.content[0].text
                logger.debug(f"API Response: {content[:200]}...")

                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """JSON 응답 검증 및 파싱"""
        try:
            # 정규식으로 JSON 배열 추출
            array_pattern = r'\[(?:[^[\]]*|\[(?:[^[\]]*|\[[^[\]]*\])*\])*\]'
            matches = list(re.finditer(array_pattern, content))

            if not matches:
                return []

            longest_match = max(matches, key=lambda match: len(match.group()))
            potential_json = longest_match.group()

            parsed = json.loads(potential_json)
            if isinstance(parsed, list):
                # 리스트의 각 항목 반복
                for item in parsed:
                    # finish_reason 키 제거
                    item.pop('finish_reason', None)
                    # 중첩된 객체 처리
                    for key, value in item.items():
                        if isinstance(value, dict):  # 값이 딕셔너리인지 확인
                            value.pop('finish_reason', None)  # 중첩된 객체에서 키 제거
                
                return parsed
            
            return []  # 파싱 실패시 빈 리스트 반환
        
        except json.JSONDecodeError:
            logger.error(f"JSON parsing failed. Response content: {content[:500]}")
            return []
        
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Chief Complaints 분류"""
        # CC를 3개의 작은 프롬프트로 분할
        cc_results = await self._make_api_call(CCPrompts.cc_analysis_prompt(texts), semaphore)
        history_results = await self._make_api_call(CCPrompts.cc_history_prompt(texts), semaphore)
        severity_results = await self._make_api_call(CCPrompts.cc_severity_prompt(texts), semaphore)
        
        # 결과 병합
        combined_results = []
        for i in range(len(texts)):
            combined_dict = {}
            if i < len(cc_results):
                combined_dict.update(cc_results[i])
            if i < len(history_results):
                combined_dict.update(history_results[i])
            if i < len(severity_results):
                combined_dict.update(severity_results[i])
            
            combined_results.append(combined_dict)
        
        return combined_results
    
    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        # API 호출 및 결과 반환
        results = await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)
        
        # 결과 로깅 (디버깅용)
        logger.debug(f"약물 분류 결과: {results}")
        
        return results

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        # API 호출 및 결과 반환
        results = await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
        logger.debug(f"장치 분류 결과: {results}")
        
        return results

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        # API 호출 및 결과 반환
        results = await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)
        logger.debug(f"습관 분류 결과: {results}")
        
        return results

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        # API 호출 및 결과 반환
        results = await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)
        logger.debug(f"찜질 분류 결과: {results}")

        return results
    
    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        # API 호출 및 결과 반환
        results = await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        logger.debug(f"마사지 분류 결과: {results}")
        
        return results 
    
    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """PI 분류 개선"""
        # 텍스트 길이 제한
        limited_texts = [text[:3000] if text and len(text) > 3000 else text for text in texts]
        
        # 더 작은 배치로 나누기
        smaller_batch_size = 10  # PI에 대해 더 작은 배치 크기 사용
        all_results = []
        
        for i in range(0, len(limited_texts), smaller_batch_size):
            batch = limited_texts[i:i+smaller_batch_size]
            
            try:
                # 각 프롬프트 개별 처리 및 로깅 추가
                logger.info(f"Processing PI basic info batch {i//smaller_batch_size + 1}")
                pi_basic_results = await self._make_api_call(PresentIllnessPrompts.pi_basic_info_prompt(batch), semaphore)
                
                logger.info(f"Processing PI examination batch {i//smaller_batch_size + 1}")
                pi_exam_results = await self._make_api_call(PresentIllnessPrompts.pi_examination_prompt(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_treatment_results = await self._make_api_call(PresentIllnessPrompts.pi_treatment_details_prompt(batch), semaphore)
                
                # 결과 병합
                for j in range(len(batch)):
                    combined_dict = {}
                    if j < len(pi_basic_results):
                        combined_dict.update(pi_basic_results[j])
                    if j < len(pi_exam_results):
                        combined_dict.update(pi_exam_results[j])
                    if j < len(pi_treatment_results):
                        combined_dict.update(pi_treatment_results[j])
                    
                    all_results.append(combined_dict)
            
            except Exception as e:
                logger.error(f"Error processing PI batch {i//smaller_batch_size + 1}: {str(e)}")
                # 오류가 발생해도 계속 진행 - 빈 결과 채우기
                all_results.extend([{} for _ in range(len(batch))])
        
        # 필요한 경우 결과 수를 원본 텍스트 수에 맞춥니다
        while len(all_results) < len(texts):
            all_results.append({})
        
        return all_results[:len(texts)]

#############################################
# 의학 데이터 처리 함수 - 이 함수를 먼저 정의해야 합니다
#############################################

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        # Process data
        processed_df = await classifier.process_all_columns(df)

        # Log statistics
        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        # Calculate success rates for each column
        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise


#############################################
# 메인 함수 - 모든 함수 정의 후에 위치
#############################################

def main():
    """메인 함수"""
    # 로깅 설정 초기화
    global logger
    logger = setup_logging()

    try:
        # 환자 데이터 로드하기
        # df = pd.read_excel('환자데이터.xlsx')
        
        # 테스트용 샘플 데이터 생성
        df_sample = df.head(10)
        
        # API 키 설정
        api_key = Config.API_KEY
        
        # 데이터 처리 - 비동기 함수 올바르게 실행
        logger.info("Starting sample data processing")
        loop = asyncio.get_event_loop()
        processed_df = loop.run_until_complete(process_medical_data(df_sample, api_key))
        
        # 결과 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f'processed_medical_data_{timestamp}.parquet'
        processed_df.to_parquet(output_file)
        logger.info(f"Data successfully saved to {output_file}")
        
        # 기본 통계 출력
        logger.info("\n=== Processing Results ===")
        for column in processed_df.columns:
            if '_' in column:  # 파생 컬럼만 표시
                valid_count = processed_df[column].notna().sum()
                logger.info(f"{column}: {valid_count} valid entries")
                
                if processed_df[column].dtype in ['object', 'category']:
                    value_counts = processed_df[column].value_counts()
                    logger.info(f"Value distribution:\n{value_counts}\n")
                    
    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        sys.exit(1)
    finally:
        logger.info("Program execution completed")


if __name__ == "__main__":
    main()

2025-03-02 11:46:02,518 - __main__ - INFO - Starting sample data processing
2025-03-02 11:46:02,529 - __main__ - INFO - Starting medical data processing at 2025-03-02 11:46:02.529199
2025-03-02 11:46:02,529 - __main__ - INFO - Processing column: CC
2025-03-02 11:46:17,423 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-03-02 11:46:22,768 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-03-02 11:46:28,226 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-03-02 11:46:28,255 - __main__ - INFO - Checkpoint saved for column CC
/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_82408/804624338.py:150: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/in

In [56]:
df_sample

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI


In [60]:
pd.read_parquet('processed_medical_data_20250302_114636.parquet')

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,CMO,MMO,Cap.pal,M.pal,Noise,Loading,Occlusion,OJ/OB,Class,Midline Shift,Deviation,CR-CO,Tongue ridging,Mucosal ridging,Ultrasono,Rt,Lt,End feel,치료계획,T-scan 악화/개선,CBCT 악화/개선,CBCT 판독소견,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration,약_medication_type,약_frequency,약_duration,약_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method"
8039,2309-172,2023-12-16,"[도착]물리치료 , 이갈이장치 del구강내과#2증상: 자다가 깨지는 않고, 오른쪽 ...","약: 많이 안먹었어요, 5개정도 먹었어요.",None,"습관: 딱딱하고 질긴 음식 안먹어요, 치아끼리 닿지 않도록 힘풀어요",찜질: 안했어요,"마사지,스트레칭: 매일 마사지, 스트레칭(샤워 후 10번)",* #14 CA,48mm --> mm after spray and stretch,50mm --> mm after spray and stretch,Rt,"Rt M +, Lt +/-, Rt tenderpoint",-,-,4567/4567,3/2,3,하악 왼 1.5,None,-,None,None,NaN,0.83 ->1.23,0.81 ->1.15,soft,"물리치료 , 장치 ck [1개월후]",NaN,NaN,NaN,오른쪽 턱,뻐근함,오른쪽 턱 뻐근함,,이갈이,"물리치료, 이갈이장치, 보톡스 주사","10년 전 입이 잘 벌어지지 않았던 과거력, 2달 전부터 입 벌릴 때 아랫입술이 오...","스트레스 받을 때 이를 악무는 습관, 딱딱하고 질긴 음식 및 얼음 섭취, 수면 시 ...","물리치료, 장치 치료, 보톡스 주사, 약물 치료(진통소염제)",1.0,0.5,None,알 수 없음,occasional,None,fair,편측성저작,high,aware,improved,both,high,10.0,self
7760,2308-60,2023-08-25,물리치료만12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07....,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,None,NaN,NaN,NaN,,,,측두하악관절 장애,,None,None,None,None,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,None
3398,2304-09,2024-02-17,"물리치료 , 장치 ck, 에식스 imp구강내과#7/ 혜련증상: 불편함 없었어요",None,장치: 격일 착용/ 불편감 X,"습관: 딱딱, 질긴음식 안먹었어요 / 치아끼리 안닿게 턱에 힘 풀어요",찜질: 얼굴에 열감있어서 못했어요,"마사지,스트레칭: 주2회 둘다 했어요",None,45mm --> mm after spray and stretch,45mm --> mm after spray and stretch,-,both) tenderpoint,rt) click 아주작게,-,-,None,s,None,None,없음,None,None,NaN,0.9 -> 1.17,1.00 -> 1.21,soft,"물리치료 , 장치 ck, APS del [2개월후]",NaN,NaN,NaN,,불편감 없음,,,,None,None,None,None,NaN,NaN,None,None,None,None,None,편측성저작,high,aware,improved,both,medium,NaN,both
21024,2202-279,2022-06-09,"구강내과#6물리치료 , 장치 ck증상 :왼쪽에서 갈리는 소리나고 좀 아팠어요장치 :...",None,None,None,None,None,장치하루빼기12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07...,37(Lt pain)mm --> mm after spray and stretch,39 mm --> mm after spray and stretch,-,None,-click / popping,-,None,None,s,None,None,None,None,None,NaN,->,->,soft,None,NaN,NaN,NaN,왼쪽,"갈리는 소리, 약간 아픔","왼쪽 갈리는 소리, 약간 아픔",,장치 착용 시 이갈이 습관,None,None,None,None,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,None
18106,2111-47,2023-05-26,"구강내과#11[도착]물리치료 , SS re-del증상: 입벌릴때 오른쪽 소리 여전,...",None,None,습관: 딱딱하고 질긴음식x.치아끼리 안 닿게 노력,찜질: 안했어요,None,None,53mm,55mm(no pain),-,Rt)M+/-,-,-,None,-,None,None,None,None,-,-,NaN,-,-,soft,24-04-04 / x-ray 6M ck,NaN,NaN,NaN,오른쪽,소리,,입벌릴 때 오른쪽 소리,,None,None,None,None,NaN,NaN,None,None,None,None,None,편측성저작,high,aware,improved,None,None,NaN,None
27930,2404-273,2024-04-29,"검색, 턱관절10년전에 한번 입이 잘안벌어졌어요시간 조금 지나고나서 다시 잘 벌어졌...",None,None,None,None,None,None,25mm --> mm after spray and stretch,33mm --> 45mm after spray and stretch,-,Both)M+(Lt>Rt),-,-,567/567,2/1,s,하오2,None,-,+,+,NaN,각져있고 안쪽이 뿌옇게 변해있는 부분O,각져있는 형태,soft,"물리치료 , SS del [2주후]",NaN,NaN,NaN,"오른쪽, 왼쪽 턱","아픔, 뻐근함","양쪽 턱 아픔, 왼쪽 주로 아픔","입벌릴 때 아랫입술 오른쪽으로 벌어짐, 왼쪽 관절 움직이는 소리",스트레스 받으면 이갈이 습관,None,None,None,None,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,None
13368,2210-203,2023-10-05,"물리치료 , 장치 ck구강내과#11증상: 지금은 불편감 전혀 없어요. (오른쪽 귀 ...",None,"장치: 위1. 아래1, 미착용 4일 / 장치 불편감X","습관: 딱딱하고 질긴 음식 잘 안먹으려고 하는데 고기류, 치아끼리 닿지 않도록 생각...","찜질: 안해요,","마사지,스트레칭: 2일에 한번 해요, 스트레칭은 5-10번정도 해요.",None,46mm -->mm after spray and stretch,46mm --> mm after spray and stretch,-,-,-,-,"왼쪽 4,5 안 닿음 -> 교합지 상에서",1.5/0.5,1,상 왼 1,S,-,+,+,NaN,1.05 -> 1.38/->1.53 / ->1.49,1.11 -> 1.49/->1.53 / ->1.31,soft,"물리치료 , 장치 ck [2개월후]",NaN,NaN,NaN,,불편감 없음,,,,None,None,None,None,5.0,5.0,10년전,None,None,None,None,편측성저작,medium,aw

In [50]:
# 컬럼명 자세히 확인
for col in df.columns:
    print(f"컬럼: '{col}', 길이: {len(col)}")

컬럼: '환자번호', 길이: 4
컬럼: '날짜', 길이: 2
컬럼: 'CC', 길이: 2
컬럼: '약', 길이: 1
컬럼: '장치 ', 길이: 3
컬럼: '습관', 길이: 2
컬럼: '찜질 ', 길이: 3
컬럼: '마사지, 스트레칭', 길이: 9
컬럼: 'PI', 길이: 2
컬럼: 'CMO', 길이: 3
컬럼: 'MMO', 길이: 3
컬럼: 'Cap.pal', 길이: 7
컬럼: 'M.pal', 길이: 5
컬럼: 'Noise', 길이: 5
컬럼: 'Loading', 길이: 7
컬럼: 'Occlusion', 길이: 9
컬럼: 'OJ/OB', 길이: 5
컬럼: 'Class', 길이: 5
컬럼: 'Midline Shift', 길이: 13
컬럼: 'Deviation', 길이: 9
컬럼: 'CR-CO', 길이: 5
컬럼: 'Tongue ridging', 길이: 14
컬럼: 'Mucosal ridging', 길이: 15
컬럼: 'Ultrasono', 길이: 9
컬럼: 'Rt', 길이: 2
컬럼: 'Lt', 길이: 2
컬럼: 'End feel', 길이: 8
컬럼: '치료계획', 길이: 4
컬럼: 'T-scan 악화/개선', 길이: 12
컬럼: 'CBCT 악화/개선', 길이: 10
컬럼: 'CBCT 판독소견', 길이: 9
